In [ ]:
import pandas as pd
import os
import numpy as np
import multiprocessing
cores = multiprocessing.cpu_count()
from IPython.display import clear_output

# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]

from nltk.tokenize import word_tokenize

# Vectorization - Doc2vec
from gensim.models.doc2vec import TaggedDocument, Doc2Vec
from gensim.test.test_doc2vec import ConcatenatedDoc2Vec

# Vectorisation - Word2Vec
from gensim.models.word2vec import Word2Vec

# Classification
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes (GaussianNB)
from sklearn.naive_bayes import GaussianNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn.svm import LinearSVC

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouches
from sklearn.neural_network import MLPClassifier

# Validation croisée - 5-fold cross-validation
stratified_kfold = StratifiedKFold(shuffle=True, random_state=42)

In [ ]:
def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

**Charger les données nettoyées**

In [ ]:
folder = '../2-scripts/1-preprocessing'
datasets = os.listdir(folder)
datasets

**Entraîner les modèles et sortir les résultats**

*Word2Vec*

In [ ]:
def vectorize(document_tokenized, model, n):
    words_vecs = [model.wv[word] for word in document_tokenized if word in model.wv]
    if len(words_vecs) == 0:
        return np.zeros(n)
        
    words_vecs = np.array(words_vecs)
    return words_vecs.mean(axis=0)

In [ ]:
for dataset in datasets:
    report = []
    ratio = int(dataset[22:-7])
    df = pd.read_excel(os.path.join(folder, dataset))

    corpus = df['cleaned_text_post'].astype('str')
    corpus = [list(tokenize_remove_stopwords(doc)) for doc in corpus]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        'Naive Bayes' : GaussianNB(), 
        'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        'Support Vector Machines (SVM)' : LinearSVC(dual='auto'),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
        # 'Random Forest': RandomForestClassifier(n_jobs = cores),
        # 'Multilayered Perceptron': MLPClassifier(max_iter = 500)
    }

    for n_features in n_features_values:
        model_w2v = Word2Vec(
                    corpus,
                    min_count= 30,
                    vector_size= n_features,
                    workers = cores,
        )

        X = np.array([vectorize(doc, model_w2v, n_features) for doc in corpus])
        print(len(X[0]))  
        y = df['category'].astype('str')
            

        for algorithm in algorithms.keys():
            # Validation croisée 
            predictions = cross_val_predict(algorithms[algorithm], X, y, cv=stratified_kfold)
            result = classification_report(y, predictions, output_dict=True)

            # Performances (rappel, précision, score F)
            results = {
                '% Incels' : int(ratio),
                'Algorithme' : algorithm,
                'Modèle de vectorisation': 'Distributed Representations (Doc2Vec)',
                'N traits discr.' : n_features,
                'accuracy': result['accuracy']    
            }
            results.update(result['macro avg'])
            report.append(results)
            
            print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
            clear_output(wait=True)

    # Exporter les résults
    # support = Nombre d'occurrence dans chaque classe
    report = pd.DataFrame(report)
    report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
    report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

    report = report.rename(columns={'support':'nb_posts_total'})
    report['nb_posts_total'] = report['nb_posts_total'].astype(int)

    # Réorganiser l'ordre des colonnes
    report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                    'Modèle de vectorisation', 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

    report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_word2vec_{ratio}pc.csv', index=False
    )

*Doc2Vec*

In [ ]:
for dataset in datasets:
    report = []
    ratio = int(dataset[22:-7])
    df = pd.read_excel(os.path.join(folder, dataset))

    corpus = df['cleaned_text_post'].astype('str')
    corpus = [list(tokenize_remove_stopwords(doc)) for doc in corpus]
    corpus = [
        TaggedDocument(words, ['{}'.format(idx)])
        for idx, words in enumerate(corpus)
    ]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        'Naive Bayes' : GaussianNB(), 
        'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        'Support Vector Machines (SVM)' : LinearSVC(dual='auto'),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
        # 'Random Forest': RandomForestClassifier(n_jobs = cores),
        # 'Multilayered Perceptron': MLPClassifier(max_iter = 500)
    }

    for n_features in n_features_values:
        print(str(int(n_features/2)))
        model_dbow = Doc2Vec(
            corpus,
            dm = 0, # DBOW Algorithm
            vector_size= int(n_features/2),
            dm_concat = 0,
            dm_mean = 1,
            workers = cores,
        )

        model_dmm = Doc2Vec(
            corpus,
            dm = 1, # Distributed Memory Mean
            vector_size = int(n_features/2),
            dm_concat = 0,
            dm_mean = 1,
            workers = cores,
        )
        
        model = ConcatenatedDoc2Vec([model_dbow, model_dmm])

        X = [model.dv[str(doc.tags[0])] for doc in corpus]
        print(len(X[0]))  
        y = df['category'].astype('str')
            

        for algorithm in algorithms.keys():
            # Validation croisée 
            predictions = cross_val_predict(algorithms[algorithm], X, y, cv=stratified_kfold)
            result = classification_report(y, predictions, output_dict=True)

            # Performances (rappel, précision, score F)
            results = {
                '% Incels' : int(ratio),
                'Algorithme' : algorithm,
                'Modèle de vectorisation': 'Distributed Representations (Doc2Vec)',
                'N traits discr.' : n_features,
                'accuracy': result['accuracy']    
            }
            results.update(result['macro avg'])
            report.append(results)
            
            print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
            clear_output(wait=True)

    # Exporter les résults
    # support = Nombre d'occurrence dans chaque classe
    report = pd.DataFrame(report)
    report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
    report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

    report = report.rename(columns={'support':'nb_posts_total'})
    report['nb_posts_total'] = report['nb_posts_total'].astype(int)

    # Réorganiser l'ordre des colonnes
    report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                    'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

    report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_doc2vec_{ratio}pc.csv', index=False
    )